# Lab 8: Relative Risks, Odds Ratios, and Logistic Regression

You'll start with two examples from Lecture 15: meat consumption and colon cancer, and sweetener use and bladder cancer. After comparing risks and odds in these tables, you'll use the diabetes data from Lecture 16 to study how diabetes status varies with BMI.

Keep `diabetes.csv` and the setup files in the same folder as this notebook.


In [ ]:
options(repr.plot.width=10, repr.plot.height=5)
source("lab8_packages.R")
if (lab8_ensure_packages()) library(ottr)
diabetes_lab8 <- read.csv("diabetes.csv")

rr_counts_lab8 <- matrix(c(59,441,50,450),nrow=2,byrow=TRUE,
    dimnames=list(c("eat_meats_yes","eat_meats_no"),c("colon_cancer_yes","colon_cancer_no")))
meat_df_lab8 <- data.frame(
    eat_meats=c(rep(1,500),rep(0,500)),
    colon_cancer=c(rep(1,59),rep(0,441),rep(1,50),rep(0,450)))
sweetener_cc_lab8 <- matrix(c(33,67,25,75),nrow=2,byrow=TRUE,
    dimnames=list(c("bladder_cancer_yes","bladder_cancer_no"),c("sweetener_yes","sweetener_no")))


## Part 1: Relative Risks and Odds Ratios

We'll start with a small table based on the meat-consumption example from Lecture 15. The rows separate the exposed and unexposed groups, and the columns show whether colon cancer occurred. The table uses simplified counts for this exercise.


In [ ]:
# Run this cell to display the table
rr_counts_lab8


**Question 1.1.** Write `table_risks_lab8(counts)` to calculate the two risks and their relative risk from a $2\times2$ count table. The first row is the exposed group and the second is the unexposed group; the first column is outcome and the second is no outcome.

Return a named numeric vector with `exposed` and `unexposed` risks and their ratio, `relative_risk`. Assume both groups contain people and at least one person has the outcome. If the unexposed risk is zero, return `Inf`.


_Points:_ 1

In [ ]:
table_risks_lab8 <- function(counts) {
    # YOUR CODE HERE
}


In [ ]:
meat_summary_q11 <- table_risks_lab8(rr_counts_lab8)
meat_risks_q11 <- meat_summary_q11[c("exposed","unexposed")]
meat_rr_q12 <- unname(meat_summary_q11["relative_risk"])
meat_summary_q11


In [ ]:
. = ottr::check("tests/q1_1.R")

**Question 1.2.** The relative risk in the table above is 1.18. What does that mean? Set `q1_2_ans` to your answer number.

1. The exposed group has an estimated risk 18% higher than the unexposed group.
2. The exposed group's estimated risk is 18 percentage points higher.
3. Exactly 18% of exposed people have colon cancer.
4. The table proves that meat consumption caused the difference.


_Points:_ 0.5

In [ ]:
q1_2_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_2.R")

**Question 1.3.** Another study reports a relative risk of 0.72. How should we interpret it? Set `q1_3_ans` to your answer number.

1. The outcome occurs in exactly 72% of exposed people.
2. The exposed group has about 28% lower risk than the unexposed group.
3. The exposed group has 72 percentage points less risk.
4. Each exposed person has an outcome probability of exactly 0.72.


_Points:_ 0.5

In [ ]:
q1_3_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_3.R")

**Question 1.4.** Now calculate relative risk from data with one row per person. Write `calc_rr_lab8(study_df)` using the binary columns `eat_meats` (1 = exposed) and `colon_cancer` (1 = outcome).

Return the risk among exposed people divided by the risk among unexposed people. Both groups are present and at least one person has the outcome. Rows and columns may be in any order. If the unexposed risk is zero, return `Inf`.


_Points:_ 1.5

In [ ]:
calc_rr_lab8 <- function(study_df) {
    # YOUR CODE HERE
}


In [ ]:
meat_rr_q15 <- calc_rr_lab8(meat_df_lab8)
meat_rr_q15


In [ ]:
. = ottr::check("tests/q1_4.R")

**Question 1.5.** Suppose we duplicate every exposed person's record in `meat_df_lab8` and keep each unexposed person's record once. How does the relative risk change? Set `q1_5_ans` to your answer number.

1. It doubles because the exposed group is twice as large.
2. It stays the same because each group's outcome proportion stays the same.
3. It is halved because the total sample size increases.
4. It cannot be calculated when the groups have different sizes.


_Points:_ 0.5

In [ ]:
q1_5_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_5.R")

### A permutation test

We'll test for an association by shuffling labels and recalculating the relative risk, as in Lecture 15. The hypotheses are $H_0: RR=1$ and $H_A: RR\ne1$. Each shuffle keeps the exposure-group sizes and the total number of cancer outcomes the same.


**Question 1.6.** Write `permute_rr_lab8(study_df, reps=2000)` to generate `reps` relative risks under the null. For each replicate, shuffle one binary column relative to the other and recalculate the relative risk. Return the results as a numeric vector.

Both exposure groups and both outcome values are present in `study_df`. A shuffle can put all outcomes in the exposed group, giving a relative risk of `Inf`. Keep those values in your results.


_Points:_ 2

In [ ]:
permute_rr_lab8 <- function(study_df,reps=2000) {
    # YOUR CODE HERE
}


In [ ]:
set.seed(152)
perm_rr_q16 <- permute_rr_lab8(meat_df_lab8,reps=2000)


In [ ]:
. = ottr::check("tests/q1_6.R")

**Question 1.7.** For a two-sided test, count departures in either direction. As in the absolute-difference permutation tests from earlier lectures, use $|RR-1|$ to measure distance from the null value.

Write `rr_perm_pvalue_lab8(null_rr, observed_rr)` to return the fraction of values in `null_rr` whose distance from 1 is **at least** that of `observed_rr`. Include any `0` or `Inf` values in the calculation. The cell below uses your function for the meat study.


_Points:_ 1.5

In [ ]:
rr_perm_pvalue_lab8 <- function(null_rr,observed_rr) {
    # YOUR CODE HERE
}


In [ ]:
perm_pval_q17 <- rr_perm_pvalue_lab8(perm_rr_q16,meat_rr_q15)
perm_pval_q17


In [ ]:
. = ottr::check("tests/q1_7.R")

### A case-control study

Now change how the sample was collected. The sweetener study from Lecture 15 selected 100 people with bladder cancer and 100 controls. The numbers with and without cancer were fixed by the study design.


In [ ]:
sweetener_cc_lab8


**Question 1.8.** Why can't we use `sweetener_cc_lab8` to calculate the population relative risk directly? Set `q1_8_ans` to your answer number.

1. Cases and controls must have the same population probability of sweetener use.
2. The researchers chose how many cases and controls to sample, so the table does not tell us the population disease risk in each exposure group.
3. Relative risk only works with continuous, normally distributed variables.
4. Case-control studies cannot estimate any probabilities or associations.


_Points:_ 0.5

In [ ]:
q1_8_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_8.R")

**Question 1.9.** Write `odds_summary_lab8(counts)` for a $2\times2$ table with cases and controls in the rows, and exposed and unexposed people in the columns. Assume all four counts are positive.

Return a named numeric vector with:

- `cases_odds`: the odds of exposure among cases;
- `controls_odds`: the odds of exposure among controls;
- `odds_ratio`: the cases' exposure odds divided by the controls' exposure odds;
- `cross_product_or`: the same odds ratio calculated directly from the four counts.


_Points:_ 1.5

In [ ]:
odds_summary_lab8 <- function(counts) {
    # YOUR CODE HERE
}


In [ ]:
sweetener_summary_q19 <- odds_summary_lab8(sweetener_cc_lab8)
sweetener_summary_q19


In [ ]:
. = ottr::check("tests/q1_9.R")

**Question 1.10.** Suppose we use this table to compare the **sample** odds of cancer in sweetener users and non-users. Keep the yes/no category order the same. How does the odds ratio compare with the one from Question 1.9? Set `q1_10_ans` to your answer number.

1. It stays the same: transposing the table leaves the cross-product ratio unchanged.
2. It becomes the reciprocal whenever the roles of exposure and outcome are exchanged.
3. It becomes the relative risk from the meat study.
4. It must equal 1 because both calculations use the same counts.


_Points:_ 0.5

In [ ]:
q1_10_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_10.R")

## Part 2: Logistic Regression with BMI

The outcome is still binary, but now the predictor, BMI, is numeric. Using the diabetes data from Lecture 16, we'll fit a logistic model to see how diabetes status varies with BMI. The plot below shows both a straight-line fit and a logistic fit.


In [ ]:
local({
    old_par <- par(mar=c(4.5,4.5,3,1))
    on.exit(par(old_par))
    bmi_grid <- data.frame(BMI=seq(min(diabetes_lab8$BMI),max(diabetes_lab8$BMI),length.out=200))
    linear_values <- predict(lm(Outcome~BMI,data=diabetes_lab8),bmi_grid)
    logistic_values <- predict(glm(Outcome~BMI,data=diabetes_lab8,family=binomial()),bmi_grid,type="response")
    plot(diabetes_lab8$BMI,diabetes_lab8$Outcome,pch=19,col=adjustcolor("steelblue",.25),
         xlab="BMI",ylab="Outcome (0 = no, 1 = yes)",
         main="Diabetes outcome and BMI",ylim=range(c(-.1,1.1,linear_values)))
    abline(h=c(0,1),lty=3,col="grey70")
    lines(bmi_grid$BMI,linear_values,col="firebrick",lwd=2)
    lines(bmi_grid$BMI,logistic_values,col="darkorange",lwd=2)
    legend("topleft",legend=c("Linear fit","Logistic fit"),col=c("firebrick","darkorange"),lwd=2,bty="n")
})


**Question 2.1.** Why might we choose logistic regression for this binary outcome? Set `q2_1_ans` to your answer number.

1. Least squares must be more accurate because the outcomes are 0 and 1.
2. Its fitted probabilities stay between 0 and 1, whereas a straight line can predict outside that range.
3. It makes BMI normally distributed, which is required before any regression can be fit.
4. It must be used whenever the predictor is continuous.


_Points:_ 0.5

In [ ]:
q2_1_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_1.R")

**Question 2.2.** Write `fit_logistic(data)` to fit `Outcome ~ BMI` with a binomial family and logit link:
$$\log\left(\frac{\widehat p}{1-\widehat p}\right)=\widehat\beta_0+\widehat\beta_1\,\texttt{BMI}.$$

Return a list with:

- `model`: the fitted `glm` object;
- `coefficients`: its named coefficient vector, with entries `"(Intercept)"` and `"BMI"`;
- `bmi_inference`: a named numeric vector with `se` and `p_value` from the BMI row of `summary(model)$coefficients`.

The input has numeric `BMI` values and `Outcome` coded as 0 or 1. Assume the fit has finite coefficient estimates. Your model should also work for predictions at new BMI values.


_Points:_ 2

In [ ]:
fit_logistic <- function(data) {
    # YOUR CODE HERE
}


In [ ]:
diabetes_fit_lab8 <- fit_logistic(diabetes_lab8)
diabetes_glm_q22 <- diabetes_fit_lab8$model
logit_coeffs_q23 <- diabetes_fit_lab8$coefficients
bmi_inference_q231 <- diabetes_fit_lab8$bmi_inference
summary(diabetes_glm_q22)


In [ ]:
. = ottr::check("tests/q2_2.R")

**Question 2.3.** What does the intercept tell us in this model? Set `q2_3_ans` to your answer number.

1. The fitted log-odds of the outcome at BMI = 0.
2. The fitted probability at the mean BMI.
3. The change in probability for a one-unit increase in BMI.
4. The average log-odds across all observations.


_Points:_ 0.5

In [ ]:
q2_3_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_3.R")

The intercept describes the model's log-odds at BMI = 0. That isn't a meaningful BMI for a person, even though zeros appear in the data file.


**Question 2.3.1.** What null hypothesis does the BMI coefficient's p-value test? Set `q2_3_1_ans` to your answer number.

1. The mean BMI is zero.
2. The population BMI coefficient in this logistic model is zero.
3. Every person has a diabetes probability of 0.5.
4. The intercept and slope are both zero.


_Points:_ 0.5

In [ ]:
q2_3_1_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_3_1.R")

**Question 2.3.2.** The model reports a small p-value for BMI. What can we conclude? Set `q2_3_2_ans` to your answer number.

1. BMI cannot be associated with the outcome because fitted probabilities stay between 0 and 1.
2. The model gives evidence of an association between BMI and the outcome's log-odds in these data.
3. The p-value proves that increasing a person's BMI causes diabetes.
4. The negative intercept means BMI should be removed from the model.


_Points:_ 0.5

In [ ]:
q2_3_2_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_3_2.R")

**Question 2.4.** Write `odds_multiplier(model, change=1)` to find the fitted odds multiplier when BMI changes by `change` units. Use the simple logistic model passed to your function and return one positive number. The change can be positive, zero, or negative.


_Points:_ 1.5

In [ ]:
odds_multiplier <- function(model, change=1) {
    # YOUR CODE HERE
}


In [ ]:
bmi_odds_mult_q24 <- odds_multiplier(diabetes_glm_q22)
bmi_odds_mult_q24


In [ ]:
. = ottr::check("tests/q2_4.R")

**Question 2.5.** Write `predict_probability(model, bmi)` to calculate fitted outcome probabilities at the BMI values in `bmi`. Return a numeric vector with one probability per input value, in the same order. Your function should work with any fitted model of the form used here.


_Points:_ 1.5

In [ ]:
predict_probability <- function(model, bmi) {
    # YOUR CODE HERE
}


In [ ]:
pred_prob_q25_35 <- predict_probability(diabetes_glm_q22,c(25,35))
names(pred_prob_q25_35) <- c("BMI25","BMI35")
pred_prob_q25_35


In [ ]:
. = ottr::check("tests/q2_5.R")

**Question 2.6.** At what BMI does the fitted probability reach `p`? Write `bmi_at_probability(model, p=0.5)` to find it from the model's log-odds equation. Return one number, assuming a nonzero BMI coefficient and `0 < p < 1`.

The fitted curve may give an answer outside the observed BMI range.


_Points:_ 1.5

In [ ]:
bmi_at_probability <- function(model, p=.5) {
    # YOUR CODE HERE
}


In [ ]:
bmi_half_prob_q26 <- bmi_at_probability(diabetes_glm_q22)
bmi_half_prob_q26


In [ ]:
. = ottr::check("tests/q2_6.R")

**Question 2.7.** Suppose two BMI values differ by 10. Which expression gives the fitted odds at the higher BMI divided by the fitted odds at the lower BMI? Set `q2_7_ans` to your answer number.

1. $10\exp(\widehat\beta_1)$
2. $\exp(10\widehat\beta_1)$
3. $\widehat p(35)-\widehat p(25)$
4. $\widehat p(35)/\widehat p(25)$


_Points:_ 0.5

In [ ]:
q2_7_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_7.R")

In [ ]:
or_bmi_35_vs_25_q27 <- odds_multiplier(diabetes_glm_q22,10)
or_bmi_35_vs_25_q27


**Question 2.8.** How should we interpret `bmi_odds_mult_q24`? Set `q2_8_ans` to your answer number.

1. The factor multiplying the fitted odds for each additional BMI unit.
2. The amount added to the fitted probability for each additional BMI unit.
3. The factor multiplying relative risk at every starting BMI.
4. The fitted odds for a person with BMI equal to 1.


_Points:_ 1

In [ ]:
q2_8_ans <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_8.R")

---

## You're finished!


To submit your work:

1. Restart the kernel and run all cells from top to bottom. In Jupyter, choose `Kernel -> Restart Kernel and Run All Cells...`, or the equivalent in your interface.
2. Make sure the notebook runs without errors and every grader check passes. If something fails, fix the first error and rerun from the beginning.
3. Save the notebook, download the `.ipynb` file, and upload it to Gradescope.
4. Wait for grading to finish and check the results on Gradescope too.
5. Confirm that your submission appears on Gradescope, and keep the confirmation email if you receive one.
